# higgsfold: soliton & local-topology analysis of a protein backbone

Analyze any protein structure with the discrete Frenet frame / Z2 gauge / soliton framework of
A. J. Niemi and co-workers (Begun, Chernodub, Molochkov & Niemi, *Phys. Rev. E* **111**, 024406 (2025)).
The backbone is described by bond angles κ and torsions τ of the Cα trace. A Z2 gauge symmetry, analogous
to symmetry breaking in an Abelian Higgs model, splits the chain into domains, and loops appear as kinks
(dark solitons) of a discrete nonlinear Schrödinger equation.

**Inputs:** a PDB ID, an AlphaFold UniProt accession, or an uploaded .pdb/.cif file.

In [ ]:
# Install. Set GITHUB_USER to the account that hosts the repository.
GITHUB_USER = "mantiswriter"  #@param {type:"string"}
assert GITHUB_USER, "Set GITHUB_USER above (the GitHub account that owns the higgsfold repository)."
!pip -q install gemmi py3Dmol numba
!pip -q install git+https://github.com/{GITHUB_USER}/higgsfold.git

In [ ]:
#@title Choose a protein
SOURCE = "1UBQ"   #@param {type:"string"}
CHAIN = ""         #@param {type:"string"}
WALL_PENALTY = 1.0 #@param {type:"number"}
MODEL = "two-field" #@param ["two-field", "paper"]
FAST = False #@param {type:"boolean"}
# To upload your own file instead:  from google.colab import files; SOURCE = list(files.upload())[0]

In [ ]:
import higgsfold as hf
from higgsfold import plots
res = hf.analyze(SOURCE, CHAIN or None, wall_penalty=WALL_PENALTY, model=MODEL,
                 global_nfev=300 if FAST else 1500, verbose=True)
print(f"{res['name']} chain {res['chain']}: {res['n_residues']} residues, {res['n_domain_walls']} domain walls")
print("reconstruction check (should be ~0):", res['reconstruction_check_rmsd'])
print("topology:", res['topology'])
print(f"soliton model: {res['n_solitons']} solitons, backbone RMSD {res['soliton_model_rmsd']:.2f} Å")

In [ ]:
plots.profiles(res); plots.landscape(res); plots.correlation_plot(res['C_k']);

### 3D view coloured by Z2 domain (sign of the gauge-fixed bond angle)

In [ ]:
import py3Dmol, numpy as np
p = res['_per_residue']; sign = np.sign(np.nan_to_num(p['kappa_gauged']))
view = py3Dmol.view(width=600, height=450)
view.addModel(open(hf.io.load_ca(SOURCE, CHAIN or None)['path']).read(), 'cif' if SOURCE.endswith('cif') or len(SOURCE)==4 else 'pdb')
view.setStyle({'cartoon': {'color': 'lightgrey'}})
for r, s, w in zip(p['resnum'], sign, p['domain_wall']):
    view.addStyle({'resi': int(r)}, {'cartoon': {'color': '#1f5f99' if s > 0 else '#d4772a'}})
    if w: view.addStyle({'resi': int(r), 'atom': 'CA'}, {'sphere': {'radius': 1.0, 'color': 'black'}})
view.zoomTo(); view.show()

### Per-residue table and downloads

In [ ]:
import pandas as pd
df = pd.DataFrame({k: v for k, v in res['_per_residue'].items()})
hf.write_outputs(res, 'higgsfold_out'); df.head(20)

### Optional: perestroika statistics over an MD trajectory
Upload a topology + trajectory (e.g. PSF/DCD, GRO/XTC) and run:
```python
!pip -q install MDAnalysis
from higgsfold.io import load_trajectory_ca
from higgsfold.ensemble import analyze_ensemble
e = analyze_ensemble(load_trajectory_ca('top.psf', 'traj.dcd'))
```